In [ ]:
import os
os.environ["ARCH_DISABLE_NUMBA"] = "1"

import pandas as pd
import yfinance as yfin
import datetime as dt
import numpy as np
import arch
import matplotlib.pyplot as plt

In [ ]:
Ticker = ['^NSEI']

# Indicate the start and end dates
start = dt.datetime.now().replace(year=dt.datetime.now().year - 7)
end = dt.datetime.now()

nifty = yfin.download(Ticker, start = start, end = end)
print(nifty)

In [ ]:
dailyReturns = nifty['Close'].pct_change(1) * 100
dailyReturns = dailyReturns.dropna()

logReturns = np.log(nifty['Close']).diff()
logReturns = logReturns.dropna()

logReturns

In [ ]:
logReturns['Returns'] = logReturns['^NSEI'] * 100

In [ ]:
returns = logReturns['Returns']

In [ ]:
model = arch.arch_model(returns, vol = 'Garch', p = 1, q = 1)
res = model.fit()

In [ ]:
print(res.summary())

In [ ]:
res.conditional_volatility

garchVol = res.conditional_volatility

In [ ]:
rolling_vol = returns.rolling(window=30).std()
rolling_vol

In [ ]:
ann = np.sqrt(252)  # trading days per year, to annualize daily volatility

plt.figure(figsize=(12, 6))
plt.plot(rolling_vol * ann, label='30-Day Rolling Volatility', color='steelblue')
plt.plot(pd.Series(garchVol, index=returns.index) * ann,
         label='GARCH(1,1) Conditional Volatility', color='red', alpha=0.8)
plt.title('30-Day Rolling vs GARCH(1,1) Volatility (Annualized)')
plt.xlabel('Date')
plt.ylabel('Annualized Volatility (%)')
plt.legend()
plt.show()

### Returns are fat-tailed — refit with a Student's-t innovation

The Normal GARCH above assumes Gaussian shocks, but Nifty returns have heavy tails (the excess kurtosis was clear in the earlier notebooks). A Student's-t innovation fits the tails better and, crucially, stops us from *understating* tail risk. The estimated degrees-of-freedom parameter `nu` is itself a fat-tail diagnostic — a smaller `nu` means heavier tails.

In [ ]:
model_t = arch.arch_model(returns, vol='Garch', p=1, q=1, dist='StudentsT')
res_t = model_t.fit(disp='off')
print(res_t.summary())
print(f"\nNormal AIC: {res.aic:.1f}    Student-t AIC: {res_t.aic:.1f}")
print(f"Estimated degrees of freedom (nu): {res_t.params['nu']:.2f}")

### Out-of-sample volatility forecast

Where GARCH earns its keep: forecasting the next few days' volatility. We forecast the conditional variance 5 trading days ahead and convert it to daily (and annualized) volatility.

In [ ]:
forecast = res_t.forecast(horizon=5, reindex=False)
daily_vol = np.sqrt(forecast.variance.iloc[-1])
print("5-day-ahead volatility forecast:")
for i, v in enumerate(daily_vol, 1):
    print(f"  t+{i}:  daily vol = {v:.3f}%   (annualized ≈ {v * np.sqrt(252):.1f}%)")

### Did the model remove the volatility clustering?

If GARCH(1,1) captured the conditional heteroskedasticity, the **standardized residuals** should have no ARCH effect left. We test with the ARCH-LM test (Engle): a *high* p-value means no remaining clustering, i.e. the model is adequate.

In [ ]:
from statsmodels.stats.diagnostic import het_arch

std_resid = (res_t.resid / res_t.conditional_volatility).dropna()
lm_stat, lm_p, f_stat, f_p = het_arch(std_resid, nlags=10)
print(f"ARCH-LM test on standardized residuals (10 lags):")
print(f"  LM statistic = {lm_stat:.2f},  p-value = {lm_p:.3f}")
print("  -> p > 0.05: no ARCH effect remains, GARCH(1,1) is adequate."
      if lm_p > 0.05 else
      "  -> p < 0.05: clustering remains, consider a higher order or a different vol model.")